# Silver — ecommerce_itens_pedido

Este notebook lê a Bronze Delta `squad1.bronze_ecommerce_itens_pedido`, aplica as 10 regras de qualidade da tabela de itens de pedido, grava a Silver Delta e registra os resultados na tabela compartilhada `squad1.dq_monitoring_logs`.




## Imports e parâmetros

In [0]:



from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from functools import reduce
import uuid
import pyarrow as pa
from deltalake import DeltaTable
from deltalake.writer import write_deltalake
from azure.core.exceptions import ResourceNotFoundError

RUN_ID = str(uuid.uuid4())

# Flags de execução
FORCAR_REPROCESSAMENTO = True
SOBRESCREVER_SILVER = True

CONTAINER_DESTINO = "squad1"
TABELA_DQ_LOGS = "squad1.dq_monitoring_logs"

container_squad1 = service_client.get_file_system_client("squad1")

print("RUN_ID:", RUN_ID)

ENTIDADE = "ecommerce_itens_pedido"
CAMINHO_BRONZE = "az://squad1/bronze/ecommerce_itens_pedido"
CAMINHO_BRONZE_PEDIDOS = "az://squad1/bronze/ecommerce_pedidos"
CAMINHO_BRONZE_PRODUTOS = "az://squad1/bronze/ecommerce_produtos"
CAMINHO_SILVER = "az://squad1/silver/ecommerce_itens_pedido"
PASTA_SILVER = "silver/ecommerce_itens_pedido"
NOME_TABELA_DQ = "ecommerce_itens_pedido"
print("Bronze:", CAMINHO_BRONZE)
print("Silver:", CAMINHO_SILVER)


## Funções auxiliares

In [0]:

def delta_existe(caminho_delta):
    try:
        DeltaTable(caminho_delta, storage_options=storage_options)
        return True
    except Exception:
        return False


def ler_delta_spark(caminho_delta):
    dt = DeltaTable(caminho_delta, storage_options=storage_options)
    pdf = dt.to_pyarrow_table().to_pandas()
    if len(pdf) == 0:
        raise Exception(f"Delta existe, mas está vazio: {caminho_delta}")
    return spark.createDataFrame(pdf)


def salvar_delta_spark(df, caminho_delta, mode="append", partition_by=None, pasta_relativa=None):
    qtd = df.count()
    if qtd == 0:
        print(f"Gravação ignorada em {caminho_delta}: DataFrame vazio.")
        return

    if mode == "overwrite" and pasta_relativa:
        try:
            container_squad1.delete_directory(pasta_relativa)
            print(f"Delta antigo removido: {pasta_relativa}")
        except ResourceNotFoundError:
            print(f"Delta antigo não existia: {pasta_relativa}")
        except Exception as e:
            print(f"Aviso ao remover {pasta_relativa}: {e}")

    table = pa.Table.from_pandas(df.toPandas(), preserve_index=False)

    kwargs = {
        "table_or_uri": caminho_delta,
        "data": table,
        "mode": mode,
        "storage_options": storage_options
    }
    if partition_by:
        kwargs["partition_by"] = partition_by

    write_deltalake(**kwargs)
    print(f"Delta gravado em {caminho_delta} | modo={mode} | registros={qtd}")


def anti_duplicidade_por_arquivo(df_novo, caminho_delta_destino, coluna_arquivo="bronze_source_file"):
    if FORCAR_REPROCESSAMENTO:
        print("FORCAR_REPROCESSAMENTO=True: todos os registros da Bronze serão avaliados novamente.")
        return df_novo

    if delta_existe(caminho_delta_destino):
        df_destino = ler_delta_spark(caminho_delta_destino)
        if coluna_arquivo in df_destino.columns:
            arquivos_processados = df_destino.select(coluna_arquivo).dropDuplicates()
            return df_novo.join(arquivos_processados, on=coluna_arquivo, how="left_anti")
    return df_novo


schema_dq_logs = StructType([
    StructField("run_id", StringType(), False),
    StructField("tabela", StringType(), False),
    StructField("regra", StringType(), False),
    StructField("status", StringType(), False),
    StructField("severidade", StringType(), False),
    StructField("qtd_registros_falhos", IntegerType(), True),
    StructField("qtd_registros_total", IntegerType(), False),
    StructField("timestamp_execucao", TimestampType(), False),
    StructField("arquivo_origem", StringType(), True),
])

COLUNAS_DQ_LOGS = [f.name for f in schema_dq_logs.fields]


def padronizar_schema_dq_logs(df):
    return (
        df.select(*COLUNAS_DQ_LOGS)
        .withColumn("run_id", F.col("run_id").cast("string"))
        .withColumn("tabela", F.col("tabela").cast("string"))
        .withColumn("regra", F.col("regra").cast("string"))
        .withColumn("status", F.col("status").cast("string"))
        .withColumn("severidade", F.col("severidade").cast("string"))
        .withColumn("qtd_registros_falhos", F.col("qtd_registros_falhos").cast("int"))
        .withColumn("qtd_registros_total", F.col("qtd_registros_total").cast("int"))
        .withColumn("timestamp_execucao", F.col("timestamp_execucao").cast("timestamp"))
        .withColumn("arquivo_origem", F.col("arquivo_origem").cast("string"))
    )


def gerar_logs_por_regras(df_validado, regras, tabela_nome, qtd_total):
    logs = []
    for regra in regras:
        coluna = regra["coluna"]
        nome = regra["regra"]
        severidade = regra.get("severidade", "Critica")

        if coluna not in df_validado.columns:
            print(f"Aviso: coluna de regra não encontrada: {coluna}")
            continue

        df_log = (
            df_validado
            .filter(F.col(coluna) == True)
            .groupBy("bronze_source_file")
            .agg(F.count(F.lit(1)).cast("int").alias("qtd_registros_falhos"))
            .withColumn("run_id", F.lit(RUN_ID))
            .withColumn("tabela", F.lit(tabela_nome))
            .withColumn("regra", F.lit(nome))
            .withColumn("status", F.lit("FAIL"))
            .withColumn("severidade", F.lit(severidade))
            .withColumn("qtd_registros_total", F.lit(int(qtd_total)).cast("int"))
            .withColumn("timestamp_execucao", F.current_timestamp())
            .withColumnRenamed("bronze_source_file", "arquivo_origem")
            .select(*COLUNAS_DQ_LOGS)
        )
        logs.append(df_log)

    if not logs:
        return spark.createDataFrame([], schema_dq_logs)

    return reduce(lambda a, b: a.unionByName(b), logs)


def gravar_dq_logs(df_logs):
    df_logs = padronizar_schema_dq_logs(df_logs)
    qtd = df_logs.count()
    print("Logs novos para dq_monitoring_logs:", qtd)
    if qtd == 0:
        print("Nenhum log novo para gravar.")
        return

    (
        df_logs
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TABELA_DQ_LOGS)
    )
    print(f"Logs gravados na tabela {TABELA_DQ_LOGS}.")


def safe_read_delta(caminho_delta, nome):
    try:
        df = ler_delta_spark(caminho_delta)
        print(f"{nome}: {df.count()} registros")
        return df
    except Exception as e:
        print(f"Aviso: não foi possível ler {nome} em {caminho_delta}: {e}")
        return None


## Ler Bronze e referências

In [0]:

df_bronze = ler_delta_spark(CAMINHO_BRONZE)
if "bronze_source_file" not in df_bronze.columns:
    raise Exception("A Bronze precisa conter bronze_source_file.")

df_micro_lote = anti_duplicidade_por_arquivo(df_bronze, CAMINHO_SILVER, "bronze_source_file")
qtd_micro_lote = df_micro_lote.count()
TEM_MICRO_LOTE_NOVO = qtd_micro_lote > 0
print("Registros para processar:", qtd_micro_lote)

df_pedidos_ref = safe_read_delta(CAMINHO_BRONZE_PEDIDOS, "Bronze pedidos")
if df_pedidos_ref is None:
    df_pedidos_ref = spark.createDataFrame([], StructType([StructField("id_pedido", LongType(), True), StructField("valor_total", DoubleType(), True)]))

df_pedidos_ref = df_pedidos_ref.select("id_pedido", F.col("valor_total").cast("double").alias("valor_total_pedido")).dropDuplicates(["id_pedido"])

df_produtos_ref = safe_read_delta(CAMINHO_BRONZE_PRODUTOS, "Bronze produtos")
if df_produtos_ref is None:
    df_produtos_ref = spark.createDataFrame([], StructType([StructField("sku", StringType(), True)]))

df_produtos_ref = df_produtos_ref.select(F.col("sku").cast("string").alias("sku")).dropDuplicates().withColumn("sku_existe", F.lit(True))


## Aplicar 10 regras

In [0]:

if TEM_MICRO_LOTE_NOVO:
    w_item = Window.partitionBy("id_item_pedido")

    df_base = (
        df_micro_lote
        .withColumn("quantidade_num", F.col("quantidade").cast("double"))
        .withColumn("preco_unitario_num", F.col("preco_unitario").cast("double"))
        .withColumn("desconto_num", F.coalesce(F.col("desconto_aplicado").cast("double"), F.lit(0.0)))
        .withColumn("sku", F.col("sku").cast("string"))
        .withColumn("qtd_id_item", F.count("*").over(w_item))
    )

    df_item_calc = df_base.withColumn("valor_item_calc", (F.col("preco_unitario_num") - F.col("desconto_num")) * F.col("quantidade_num"))
    df_total_itens = df_item_calc.groupBy("id_pedido").agg(F.round(F.sum("valor_item_calc"), 2).alias("valor_total_itens"), F.count("*").alias("qtd_itens_pedido"))
    df_pedido_tem_item = df_total_itens.select("id_pedido").dropDuplicates().withColumn("pedido_tem_item", F.lit(True))

    df_regras = (
        df_item_calc
        .join(df_pedidos_ref, on="id_pedido", how="left")
        .join(df_produtos_ref, on="sku", how="left")
        .join(df_total_itens, on="id_pedido", how="left")
        .withColumn("sku_existe", F.coalesce(F.col("sku_existe"), F.lit(False)))
    )

    df_silver = (
        df_regras
        .withColumn("r1_id_item_pedido_falhou", F.col("id_item_pedido").isNull() | (F.col("qtd_id_item") > 1))
        .withColumn("r2_id_pedido_inexistente_falhou", F.col("id_pedido").isNull() | F.col("valor_total_pedido").isNull())
        .withColumn("r3_sku_inexistente_falhou", F.col("sku").isNull() | (F.col("sku_existe") == False))
        .withColumn("r4_quantidade_falhou", F.col("quantidade_num").isNull() | (F.col("quantidade_num") < 1) | (F.col("quantidade_num") != F.floor(F.col("quantidade_num"))))
        .withColumn("r5_preco_unitario_falhou", F.col("preco_unitario_num").isNull() | (F.col("preco_unitario_num") <= 0))
        .withColumn("r6_desconto_maior_preco_falhou", F.col("desconto_num") > F.col("preco_unitario_num"))
        .withColumn("r7_total_pedido_divergente_falhou", F.col("valor_total_pedido").isNull() | (F.abs(F.col("valor_total_itens") - F.col("valor_total_pedido")) > 0.01))
        .withColumn("r8_desconto_negativo_falhou", F.col("desconto_num") < 0)
        .withColumn("r9_desconto_maior_50_falhou", F.col("preco_unitario_num").isNotNull() & (F.col("preco_unitario_num") > 0) & ((F.col("desconto_num") / F.col("preco_unitario_num")) > 0.5))
        .withColumn("r10_pedido_sem_item_falhou", F.lit(False))
    )

    regras_dq = [
        {"coluna":"r1_id_item_pedido_falhou", "regra":"R1_ID_ITEM_PEDIDO_NULO_OU_DUPLICADO", "severidade":"Critica"},
        {"coluna":"r2_id_pedido_inexistente_falhou", "regra":"R2_ID_PEDIDO_INEXISTENTE", "severidade":"Critica"},
        {"coluna":"r3_sku_inexistente_falhou", "regra":"R3_SKU_INEXISTENTE", "severidade":"Critica"},
        {"coluna":"r4_quantidade_falhou", "regra":"R4_QUANTIDADE_INVALIDA", "severidade":"Critica"},
        {"coluna":"r5_preco_unitario_falhou", "regra":"R5_PRECO_UNITARIO_INVALIDO", "severidade":"Critica"},
        {"coluna":"r6_desconto_maior_preco_falhou", "regra":"R6_DESCONTO_MAIOR_QUE_PRECO", "severidade":"Critica"},
        {"coluna":"r7_total_pedido_divergente_falhou", "regra":"R7_TOTAL_ITENS_DIVERGENTE_PEDIDO", "severidade":"Critica"},
        {"coluna":"r8_desconto_negativo_falhou", "regra":"R8_DESCONTO_NEGATIVO", "severidade":"Critica"},
        {"coluna":"r9_desconto_maior_50_falhou", "regra":"R9_DESCONTO_ACIMA_50_POR_CENTO", "severidade":"Critica"},
        {"coluna":"r10_pedido_sem_item_falhou", "regra":"R10_PEDIDO_SEM_ITEM", "severidade":"Critica"},
    ]
    falha_critica = reduce(lambda a,b: a | b, [F.col(r["coluna"]) for r in regras_dq])
    df_silver = df_silver.withColumn("silver_linha_valida", ~falha_critica).withColumn("silver_processed_at", F.current_timestamp()).drop("qtd_id_item", "sku_existe")

    # Log especial R10: pedidos sem nenhum item associado.
    df_pedidos_sem_item_logs = (
        df_pedidos_ref
        .join(df_pedido_tem_item, on="id_pedido", how="left")
        .filter(F.col("pedido_tem_item").isNull())
        .agg(F.count("*").cast("int").alias("qtd_registros_falhos"))
        .withColumn("run_id", F.lit(RUN_ID))
        .withColumn("tabela", F.lit(NOME_TABELA_DQ))
        .withColumn("regra", F.lit("R10_PEDIDO_SEM_ITEM"))
        .withColumn("status", F.lit("FAIL"))
        .withColumn("severidade", F.lit("Critica"))
        .withColumn("qtd_registros_total", F.lit(int(df_pedidos_ref.count())).cast("int"))
        .withColumn("timestamp_execucao", F.current_timestamp())
        .withColumn("arquivo_origem", F.lit("ecommerce_pedidos"))
        .filter(F.col("qtd_registros_falhos") > 0)
        .select(*COLUNAS_DQ_LOGS)
    )

    display(df_silver.limit(10))
else:
    print("Sem micro-lote novo.")


## Gravar dq_monitoring_logs sem duplicidade

## Validação final

In [0]:

print("\n===== Validação final =====")
try:
    print("Silver:", CAMINHO_SILVER)
    df_val_silver = ler_delta_spark(CAMINHO_SILVER)
    print("Registros na Silver:", df_val_silver.count())
    display(df_val_silver.limit(20))
except Exception as e:
    print("Silver ainda não disponível ou vazia:", e)

try:
    df_logs_val = spark.table(TABELA_DQ_LOGS)
    print("Registros totais em dq_monitoring_logs:", df_logs_val.count())
    display(df_logs_val.orderBy(F.col("timestamp_execucao").desc()).limit(30))
except Exception as e:
    print("Não foi possível consultar dq_monitoring_logs:", e)
